# Circular-Shift Permutation GLM for 2-Photon Calcium Imaging

This notebook walks through the full analysis pipeline for identifying which neurons encode specific task variables using a circular-shift permutation GLM.

**Overview:**
1. Load data from HDF5 (MATLAB v7.3 format)
2. Build a design matrix with task regressors
3. Run the GLM with circular-shift permutation significance testing
4. Time-resolved encoding profiles
5. 2-window magnitude analysis (early vs late)
6. Population classification and visualization

**Requirements:** `numpy`, `scipy`, `h5py`, `matplotlib`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import sys
import os

# Add parent directory to path if running from notebooks/
sys.path.insert(0, os.path.abspath('..'))

from glm_permutation import (
    H5SessionReader, load_rois,
    build_full_dm, build_windowed_dm,
    compute_pvalues_circular_shift, compute_delta_r2,
    circular_shift_null,
    make_lick_rate_regressor, make_event_kernel,
    classify_neurons,
    set_standalone_style, bar_with_points, shade_windows, smooth_trace,
)

set_standalone_style()
print('Imports OK')

## 1. Configuration

Set your data path, experimental groups, and session indices below.

In [ ]:
# ============================================================
# USER CONFIG -- edit these for your experiment
# ============================================================

MAT_PATH = '/path/to/your/data.mat'  # HDF5 v7.3 format
ROOT_KEY = 'your_struct_name'         # top-level MATLAB struct name

# Session indices to analyze (0-based)
# Example: if you have Hab, Cond, Test sessions, pick the Test indices
TEST_INDICES = [5, 6, 7]

# Experimental groups: dict mapping group name -> list of subject IDs (1-based)
GROUPS = {
    'Group_A': [1, 2, 3, 4],
    'Group_B': [5, 6, 7],
}

GROUP_COLORS = {
    'Group_A': '#d62728',  # red
    'Group_B': '#1f77b4',  # blue
}

# GLM parameters
ALPHA = 0.01             # significance threshold
LICK_SIGMA = 0.15        # Gaussian kernel width for lick rate (seconds)
LICK_SHIFT_MS = 300      # +/- time shift for lick regressors (ms)
KERNEL_TYPE = 'gaussian'  # 'gaussian' or 'gcamp'

# Time windows for magnitude analysis
EARLY_WINDOW = (0.0, 2.0)  # motor/lick-dominant period
LATE_WINDOW = (3.0, 5.0)   # sensory/taste-dominant period

## 2. Load Data

Open the HDF5 file and create session readers. The `H5SessionReader` class handles the nested object-reference structure used by MATLAB's `-v7.3` save format.

**Important:** If your data is in a different format (NWB, plain HDF5, numpy arrays), you can write a custom reader class that provides the same interface:
- `.time_axis`: 1D array of timepoints
- `.n_trials`, `.n_neurons`: integers
- `.get_trial_spikes(trial_idx)`: returns (T, N) array
- `.get_trial_sol_id(trial_idx)`: returns trial type integer
- `.get_trial_lick_times(trial_idx)`: returns 1D array of event times

In [ ]:
f = h5py.File(MAT_PATH, 'r')
root = f[ROOT_KEY]
session_ds = root['subject']['session']
n_mice = session_ds.shape[0]

print(f'Loaded {MAT_PATH}')
print(f'Number of mice: {n_mice}')

# Quick look at one mouse
mi = 0  # mouse index (0-based)
readers = [H5SessionReader(f, session_ds, mi, si) for si in TEST_INDICES]
print(f'\nMouse {mi}:')
print(f'  Neurons: {readers[0].n_neurons}')
print(f'  Trials per session: {[r.n_trials for r in readers]}')
print(f'  Time axis: {readers[0].time_axis[0]:.2f} to {readers[0].time_axis[-1]:.2f} s')
print(f'  Timepoints per trial: {readers[0].n_timepoints_per_trial}')

## 3. Visualize the Regressors

Before running the GLM, inspect what the design matrix looks like for a single trial.

In [ ]:
reader = readers[0]
time_axis = reader.time_axis

# Pick a trial
trial_idx = 0
lick_times = reader.get_trial_lick_times(trial_idx)
sol_id = reader.get_trial_sol_id(trial_idx)
spikes = reader.get_trial_spikes(trial_idx)

# Build regressors for this trial
taste_kernel = make_event_kernel(time_axis, 0.0, kernel_width=3.0)
lick_reg = make_lick_rate_regressor(lick_times, time_axis,
                                     sigma=LICK_SIGMA, shift_ms=LICK_SHIFT_MS)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

# Neural activity (first 5 neurons)
ax = axes[0]
for ni in range(min(5, spikes.shape[1])):
    ax.plot(time_axis, spikes[:, ni], alpha=0.6, linewidth=0.8)
ax.set_ylabel('dF/F')
ax.set_title(f'Trial {trial_idx} (sol_id={sol_id}): Neural activity (5 example neurons)')

# Taste kernel
ax = axes[1]
ax.imshow(taste_kernel.T, aspect='auto', extent=[time_axis[0], time_axis[-1], 0, taste_kernel.shape[1]],
          cmap='Blues')
ax.set_ylabel('Taste kernel\n(time bins)')
ax.set_title(f'Taste kernel ({"active" if sol_id == 1 else "zero -- water trial"})')

# Lick rate regressors
ax = axes[2]
ax.imshow(lick_reg.T, aspect='auto', extent=[time_axis[0], time_axis[-1], 0, lick_reg.shape[1]],
          cmap='Oranges')
ax.set_ylabel('Lick rate\n(shifted copies)')
ax.set_title(f'Lick rate regressor ({len(lick_times)} licks, sigma={LICK_SIGMA}s, shifts=+/-{LICK_SHIFT_MS}ms)')

# Lick times
ax = axes[3]
if len(lick_times) > 0:
    ax.eventplot([lick_times], lineoffsets=0.5, linelengths=0.8, color='k')
ax.set_xlabel('Time from stimulus (s)')
ax.set_ylabel('Lick events')
ax.set_xlim(time_axis[0], time_axis[-1])

plt.tight_layout()
plt.show()

## 4. Run the Full-Trial GLM for One Mouse

This builds the concatenated design matrix across test sessions and runs the circular-shift permutation test. The output includes per-neuron p-values and delta-R2 for each predictor.

In [ ]:
import time as timer

mi = 0  # mouse index
readers = [H5SessionReader(f, session_ds, mi, si) for si in TEST_INDICES]
n_neurons = min(r.n_neurons for r in readers)

print(f'Building design matrix for mouse {mi} ({n_neurons} neurons)...')
t0 = timer.time()

X_list, Y, boundaries = build_full_dm(
    readers,
    sigma=LICK_SIGMA,
    shift_ms=LICK_SHIFT_MS,
    kernel_type=KERNEL_TYPE,
)

print(f'Design matrix: {sum(x.shape[1] for x in X_list)} predictor columns')
print(f'Y shape: {Y.shape} (timepoints x neurons)')
print(f'Session boundaries: {boundaries}')

# Run circular-shift permutation test
print('\nRunning circular-shift permutation GLM...')
pvals, delta_r2, sig_fracs = compute_pvalues_circular_shift(
    X_list, Y, boundaries, alpha=ALPHA
)

elapsed = timer.time() - t0
print(f'Done in {elapsed:.1f}s')

CHUNK_NAMES = ['Taste', 'Lick rate', 'Lick x Taste', 'Spout', 'Trial #', 'Session']
print(f'\nSignificant neurons (alpha={ALPHA}):')
for i, name in enumerate(CHUNK_NAMES[:len(sig_fracs)]):
    print(f'  {name}: {sig_fracs[i]:.1f}%')

## 5. Visualize Results: Delta-R2 Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, pred_idx, name, color in [
    (axes[0], 0, 'Taste', '#d62728'),
    (axes[1], 1, 'Lick rate', '#1f77b4'),
    (axes[2], 2, 'Lick x Taste', '#9467bd'),
]:
    sig_mask = pvals[:, pred_idx] < ALPHA
    ax.hist(delta_r2[:, pred_idx], bins=50, color='gray', alpha=0.5, label='All')
    if np.any(sig_mask):
        ax.hist(delta_r2[sig_mask, pred_idx], bins=50, color=color, alpha=0.8,
                label=f'Sig (n={np.sum(sig_mask)})')
    ax.set_xlabel('Delta-R2')
    ax.set_ylabel('# neurons')
    ax.set_title(name)
    ax.legend(fontsize=9)

plt.suptitle(f'Mouse {mi}: Variance uniquely explained per predictor', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Time-Resolved GLM

Slide a 1-second window across the trial to see WHEN each variable is encoded.

In [ ]:
from glm_permutation.glm_timeresolved import run_timeresolved_mouse

print(f'Running time-resolved GLM for mouse {mi}...')
t0 = timer.time()
tr_result = run_timeresolved_mouse(readers, win_size=1.0, win_step=0.25, alpha=ALPHA)
elapsed = timer.time() - t0
print(f'Done in {elapsed:.1f}s ({len(tr_result["win_centers"])} windows)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

wc = tr_result['win_centers']
pred_names = ['Taste', 'Lick rate', 'Lick x Taste']
pred_colors = ['#d62728', '#1f77b4', '#9467bd']

for pi, (name, color) in enumerate(zip(pred_names, pred_colors)):
    axes[0].plot(wc, tr_result['sig_fracs'][:, pi], color=color, linewidth=2, label=name)
    axes[1].plot(wc, tr_result['mean_dr2'][:, pi], color=color, linewidth=2, label=name)

for ax, ylabel, title in [
    (axes[0], '% significant neurons', 'Encoding prevalence'),
    (axes[1], 'Mean delta-R2 (%) among sig.', 'Effect magnitude'),
]:
    ax.set_xlabel('Time from stimulus (s)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.4)
    shade_windows(ax, early=EARLY_WINDOW, late=LATE_WINDOW)

plt.suptitle(f'Mouse {mi}: Time-resolved encoding', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Two-Window Magnitude Analysis

Compare encoding in the early (motor) vs late (sensory) window.

In [ ]:
from glm_permutation.glm_magnitude import run_magnitude_mouse

print(f'Running 2-window analysis for mouse {mi}...')
t0 = timer.time()
mag_result = run_magnitude_mouse(
    readers,
    early_window=EARLY_WINDOW,
    late_window=LATE_WINDOW,
    alpha=ALPHA,
)
elapsed = timer.time() - t0
print(f'Done in {elapsed:.1f}s')

print(f'\nEarly window {EARLY_WINDOW}:')
print(f'  Taste sig: {mag_result["early_sig_fracs"][0]:.1f}%')
print(f'  Lick sig:  {mag_result["early_sig_fracs"][1]:.1f}%')
print(f'\nLate window {LATE_WINDOW}:')
print(f'  Taste sig: {mag_result["late_sig_fracs"][0]:.1f}%')
print(f'  Lick sig:  {mag_result["late_sig_fracs"][1]:.1f}%')

## 8. Population Classification

Classify each neuron based on what it encodes in the two time windows.

In [ ]:
classes = classify_neurons(mag_result)
n_total = len(classes)

class_names = {0: 'Neither', 1: 'Lick only', 2: 'Taste only', 3: 'Both'}
class_colors = {0: '#BDBDBD', 1: '#2196F3', 2: '#FF5722', 3: '#9C27B0'}

print(f'Population classification (n={n_total} neurons):')
for cls in [1, 2, 3, 0]:
    n = np.sum(classes == cls)
    print(f'  {class_names[cls]}: {n} ({100*n/n_total:.1f}%)')

# Pie chart
fig, ax = plt.subplots(figsize=(6, 6))
sizes = [np.sum(classes == c) for c in [1, 2, 3, 0]]
labels = [f'{class_names[c]}\n({sizes[i]})' for i, c in enumerate([1, 2, 3, 0])]
colors = [class_colors[c] for c in [1, 2, 3, 0]]
ax.pie(sizes, labels=labels, colors=colors, startangle=90,
       wedgeprops=dict(width=0.5, edgecolor='white'))
ax.set_title(f'Mouse {mi}: Functional classification', fontweight='bold')
plt.show()

## 9. Run All Mice and Compare Groups

Loop over all mice, run the 2-window analysis, and compare encoding proportions across groups.

In [ ]:
from glm_permutation.glm_magnitude import run_magnitude_all

all_results = run_magnitude_all(
    f, session_ds, n_mice, TEST_INDICES, GROUPS,
    early_window=EARLY_WINDOW,
    late_window=LATE_WINDOW,
    alpha=ALPHA,
)

# Save results
np.save('magnitude_results.npy', all_results, allow_pickle=True)
print('\nResults saved to magnitude_results.npy')

In [ ]:
# Group comparison: population proportions
from glm_permutation.population_classify import population_proportions

fig, ax = plt.subplots(figsize=(8, 5))

pop_labels = ['Lick only', 'Taste only', 'Both']
pop_keys = [1, 2, 3]  # class indices
x = np.arange(len(pop_labels))
bar_w = 0.35

for offset, (gname, gids) in enumerate(GROUPS.items()):
    props = population_proportions(all_results, gids)  # (n_mice, 4)
    # Extract columns for lick-only, taste-only, both (indices 1, 2, 3)
    vals = [props[:, c] for c in pop_keys]
    bar_with_points(ax, x + offset * bar_w, vals, bar_w,
                    color=GROUP_COLORS[gname], label=gname)

ax.set_xticks(x + bar_w * (len(GROUPS) - 1) / 2)
ax.set_xticklabels(pop_labels)
ax.set_ylabel('% of neurons')
ax.set_title('Population proportions by group')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Encoding Magnitude Comparison

Compare the % of significant neurons for each predictor across windows and groups.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

categories = ['Taste\n(early)', 'Lick\n(early)', 'Taste\n(late)', 'Lick\n(late)']
x = np.arange(len(categories))
bar_w = 0.35

for offset, (gname, gids) in enumerate(GROUPS.items()):
    mice = [v for v in all_results.values()
            if isinstance(v, dict) and v.get('subject_id') in gids]
    vals_list = []
    for m in mice:
        vals_list.append([
            m['early_sig_fracs'][0],  # taste early
            m['early_sig_fracs'][1],  # lick early
            m['late_sig_fracs'][0],   # taste late
            m['late_sig_fracs'][1],   # lick late
        ])
    vals_arr = np.array(vals_list)
    per_cat = [vals_arr[:, i] for i in range(4)]
    bar_with_points(ax, x + offset * bar_w, per_cat, bar_w,
                    color=GROUP_COLORS[gname], label=gname)

ax.set_xticks(x + bar_w * (len(GROUPS) - 1) / 2)
ax.set_xticklabels(categories)
ax.set_ylabel('% significant neurons')
ax.set_title('Encoding by predictor and time window')
ax.axvline(1.5, color='gray', linestyle=':', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Clean up
f.close()
print('Done. HDF5 file closed.')

## Next Steps

- **Kernel sensitivity sweep:** Use `glm_permutation.kernel_sweep` to verify results are robust to kernel parameter choices.
- **Spatial ROI map:** Use `glm_permutation.spatial_map` to visualize neuron classes on the imaging field of view.
- **Time-resolved group comparison:** Run `glm_timeresolved.run_timeresolved_all()` for all mice and plot group-averaged temporal profiles.
- **Custom regressors:** Add your own predictor columns to `X_list` in `build_full_dm()` or write a custom design matrix builder.